# CDEaDPP 2026.2 - Trabalho Final: Zircônia

## 1. Delimitação do Problema e Desenho da Pesquisa

Este notebook é um **template educacional de protocolo empírico replicável**.
O caso didático é inteiramente simulado e toma como cenário o país imaginário
Zircônia: população grande, fortemente desigual, sistema público de saúde
pressionado por filas e presença territorial incompleta da Defensoria Pública.
O objetivo é ensinar como passar de um problema empírico delimitado para um
quadro de variáveis, dados simulados, exploração, modelagem, testes de
hipóteses e planejamento amostral.

A unidade de análise será o **pedido individual de acesso à saúde** registrado
no ZUS. O desenho é **observacional retrospectivo**: a assistência jurídica
não é sorteada pelo pesquisador; ela ocorre de modo associado a características
da pessoa, do caso e da unidade territorial. Por isso, mesmo quando a simulação
injeta um efeito gerador conhecido, a interpretação estatística no notebook
deve falar em associação ajustada, não em prova causal automática.

| Elemento T.E.M.P. | Delimitação didática | Decisão que o estudante deve adaptar |
|---|---|---|
| Tempo | Pedidos simulados entre 2023 e 2025 | Fixar janela de coleta e seguimento |
| Espaço | 20 regiões e unidades territoriais do ZUS na Zircônia | Descrever sistema de referência e cobertura |
| Matéria | Acesso à saúde pública e acesso à justiça | Definir política, direito ou serviço estudado |
| População | Pessoas de baixa renda com pedido de atendimento/intervenção em saúde | Definir população-alvo, acessível e amostrada |
| Unidade elementar | Pedido administrativo/judicial de saúde | Verificar múltiplos pedidos por pessoa |
| Exposição | Assistência jurídica institucional antes da decisão final | Definir momento e critério de exposição |
| Desfecho primário | Índice de acesso efetivo à saúde | Justificar constructo e escala |
| Desfecho secundário | Desfecho do pedido em cinco categorias nominais | Fixar categorias exaustivas e exclusivas |

**Problema de pesquisa modelo:** entre pedidos de saúde da população de baixa
renda da Zircônia, há associação entre assistência jurídica institucional e
acesso efetivo à saúde, após ajuste por gravidade clínica, renda, idade,
completude documental, região, ano e unidade do ZUS?

**Hipótese primária bilateral:** $H_0: \beta_{\text{assistência jurídica}}=0$
contra $H_1: \beta_{\text{assistência jurídica}}\ne0$ no modelo de regressão
linear para `indice_acesso_saude`.

**Hipótese secundária:** $H_0$: a distribuição de `desfecho_pedido` é igual
entre pedidos com e sem assistência jurídica. $H_1$: ao menos uma das cinco
proporções difere.

**Variáveis ocultas e confundidoras.** Letramento digital, capacidade familiar
de navegar no ZUS, qualidade prévia dos documentos, distância até a Defensoria
e capital social podem afetar simultaneamente a chance de receber assistência
jurídica e o acesso efetivo à saúde. O gerador de dados incluirá uma variável
latente de letramento digital para ilustrar confundimento residual; ela não
será exportada para a base analítica.

**Fontes de apoio usadas na construção do template.** A organização segue os
guias locais da disciplina sobre formulação de problema T.E.M.P., população,
sistema de referência, unidade amostral, escalas de medição, erros amostrais
e não amostrais, hipóteses, erros tipo I e II, validade interna, validade
externa e variáveis ocultas. Como referências bibliográficas de apoio, o
estudante deve consultar, entre outras, Bolfarine e Bussab para amostragem,
Barbetta e Agresti/Finlay para estatística social, Babbie e Bryman para
desenho de pesquisa social, Pearl/Glymour/Jewell para caminhos causais e
Bruce/Bruce para práticas de ciência de dados.

In [1]:
# Importar bibliotecas necessárias para todo o notebook.
import importlib.metadata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display
from statsmodels.stats.power import GofChisquarePower, TTestIndPower

# Fixar parâmetros inferenciais antes de observar qualquer resultado.
SEMENTE = 2026
alpha = 0.05
beta = 0.15
poder_alvo = 1 - beta
tamanho_piloto = 50

# Criar gerador principal de números pseudoaleatórios para a simulação.
rng = np.random.default_rng(SEMENTE)

# Localizar a raiz do projeto por arquivos estáveis do repositório.
candidatos_raiz = (Path.cwd(), *Path.cwd().parents)
raizes_possiveis = [
    pasta
    for pasta in candidatos_raiz
    if (pasta / "pyproject.toml").is_file()
    and (pasta / "md" / "trabalho_final_simulacao.md").is_file()
]
if not raizes_possiveis:
    raise FileNotFoundError(
        "Execute o notebook na raiz do projeto ou em uma subpasta dele."
    )
RAIZ_PROJETO = raizes_possiveis[0].resolve()

# Preparar diretórios de saída usados pelas próximas seções.
PASTA_SAIDA = (RAIZ_PROJETO / "out" / "cdedpp_2026_zirconia").resolve()
PASTA_FIGURAS = (PASTA_SAIDA / "figuras").resolve()
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Validar parâmetros que controlam inferência e reprodutibilidade.
assert isinstance(SEMENTE, int), "A semente precisa ser um inteiro."
assert 0 < alpha < 1, "alpha deve estar entre 0 e 1."
assert 0 < beta < 1, "beta deve estar entre 0 e 1."
assert np.isclose(poder_alvo, 1 - beta), "poder_alvo deve ser igual a 1 - beta."
assert tamanho_piloto == 50, "Este template exige piloto simulado com n = 50."
assert PASTA_SAIDA.is_relative_to(RAIZ_PROJETO), "Saída fora da raiz do projeto."
assert PASTA_FIGURAS.is_relative_to(PASTA_SAIDA), "Figuras fora da pasta da seção."

# Configurar estilo visual e formato de exibição para tabelas.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 80)

# Registrar versões para reprodutibilidade computacional.
pacotes = ("numpy", "pandas", "matplotlib", "seaborn", "scipy", "statsmodels")
versoes = {
    pacote: importlib.metadata.version(pacote)
    for pacote in pacotes
}
quadro_ambiente = pd.DataFrame(
    {
        "item": ["raiz", "pasta_saida", "semente", "alpha", "beta", "poder", "piloto_n"],
        "valor": [
            str(RAIZ_PROJETO),
            str(PASTA_SAIDA),
            SEMENTE,
            alpha,
            beta,
            poder_alvo,
            tamanho_piloto,
        ],
    }
)

# Exibir o registro do ambiente sem gravar outputs no arquivo-fonte.
display(quadro_ambiente)
display(pd.DataFrame(versoes.items(), columns=["pacote", "versao"]))

,item,valor
0,raiz,E:\DADOS\Documents\AmbienteDeTrabalho\Projetos...
1,pasta_saida,E:\DADOS\Documents\AmbienteDeTrabalho\Projetos...
2,semente,2026
3,alpha,0.05
4,beta,0.15
5,poder,0.85
6,piloto_n,50


,pacote,versao
0,numpy,2.5.2
1,pandas,3.0.5
2,matplotlib,3.11.1
3,seaborn,0.13.2
4,scipy,1.18.0
5,statsmodels,0.14.6


### Interpretação metodológica da seção

A semente garante que os mesmos dados simulados possam ser recuperados por
outro pesquisador. Ela não transforma o desenho em experimento aleatorizado.
O grupo com assistência jurídica continuará sendo selecionado por um mecanismo
observacional, possivelmente relacionado a gravidade, vulnerabilidade, região
e variáveis não observadas. Essa distinção é central para a validade interna.

Ao adaptar o template, o estudante deve registrar: população-alvo, população
acessível, sistema de referência, critério de inclusão, critério de exclusão,
unidade de análise, janela temporal, instrumento de medição, variáveis ocultas
plausíveis, direção esperada dos vieses e limites de generalização.

---

## 2. Dicionário de Dados e Quadro de Variáveis

A tabela abaixo é parte do protocolo, não apenas documentação posterior. Em
uma pesquisa empírica replicável, cada variável deve ter definição operacional,
domínio admissível, escala de mensuração, unidade de medida, tipo computacional
e papel analítico definidos antes da coleta ou da simulação principal.

| Nome abreviado | Significado | Tipo | Escala de medição | Unidade de medida | Tipo Python adequado | Papel | Exemplos |
|---|---|---|---|---|---|---|---|
| `id_pedido` | Identificador artificial único do pedido de acesso à saúde | Texto | Identificador sem escala inferencial | Não se aplica | `string` | Chave | `ZIR-000001`, `ZIR-000250` |
| `assistencia_juridica` | Recebeu assistência jurídica institucional antes da decisão final | Categórica binária | Nominal binária | Categoria Sim/Não | `category` | Independente principal | `Sim`, `Não` |
| `indice_acesso_saude` | Escore composto de acesso efetivo ao cuidado em saúde | Numérica contínua | Intervalar assumida | Pontos de 0 a 100 | `float64` | Dependente primária | `41.7`, `68.2` |
| `desfecho_pedido` | Resultado final observado na janela retrospectiva | Categórica multinomial | Nominal com cinco níveis | Categoria | `category` | Dependente secundária | `internacao_realizada`, `negado` |
| `tempo_espera_dias` | Tempo entre o protocolo do pedido e a decisão ou atendimento | Numérica contínua | Razão | Dias | `float64` | Desfecho auxiliar/covariável descritiva | `12.5`, `83.0` |
| `gravidade_clinica` | Gravidade clínica registrada antes da assistência | Categórica ordinal | Ordinal | Nível 1 a 4 | `Int64` ou `category` ordenada | Confundidora observada | `1`, `4` |
| `renda_pc_zirc` | Renda familiar mensal per capita do requerente | Numérica contínua | Razão | Z$ por pessoa/mês | `float64` | Confundidora observada | `180.50`, `940.00` |
| `idade` | Idade do requerente no protocolo | Numérica discreta | Razão | Anos completos | `Int64` | Confundidora observada | `34`, `72` |
| `regiao_zirconia` | Região administrativa da Zircônia | Categórica multinomial | Nominal | Região | `category` | Estrato/contexto | `R03`, `R17` |
| `unidade_zus` | Unidade territorial do ZUS vinculada ao pedido | Categórica multinomial | Nominal | Unidade administrativa | `category` | Cluster inferencial | `UZ-014`, `UZ-087` |
| `capilaridade_defensoria` | Presença local da Defensoria Pública antes do pedido | Categórica ordinal | Ordinal | Nível 0 a 3 | `Int64` ou `category` ordenada | Confundidora/contexto | `0`, `3` |
| `completude_documental` | Documentação inicial suficiente para análise do pedido | Categórica binária | Nominal binária | Categoria Sim/Não | `category` | Covariável prévia | `Sim`, `Não` |
| `ano_pedido` | Ano de protocolo do pedido | Temporal discreta | Calendário | Ano | `Int64` | Contexto temporal | `2023`, `2025` |
| `letramento_digital_latente` | Facilidade não observada de usar sistemas do ZUS e buscar canais institucionais | Latente binária | Nominal binária | Não observada | Não exportar | Confundidora oculta | `0`, `1` apenas no gerador |

**Ressalva sobre escala intervalar.** O `indice_acesso_saude` será tratado
como intervalar porque diferenças de pontos serão interpretadas como
comparáveis. Isso é uma hipótese substantiva e psicométrica do instrumento,
não uma consequência automática do uso de números decimais. Em dados reais,
o estudante deve justificar itens, pesos, consistência e evidências de validade.

**Ressalva sobre escala multinomial.** `desfecho_pedido` terá cinco categorias
nominais exaustivas e exclusivas. A ordem de apresentação é didática e não
autoriza calcular médias dos códigos nem tratar o desfecho como ordinal sem
argumento teórico independente.

In [2]:
# Definir categorias nominais do desfecho antes da simulação.
CATEGORIAS_DESFECHO = [
    "internacao_realizada",
    "tratamento_ambulatorial",
    "fornecimento_medicamento",
    "negado",
    "pendente_ou_sem_execucao",
]

# Definir categorias binárias padronizadas para variáveis Sim/Não.
CATEGORIAS_BINARIAS = ["Não", "Sim"]

# Listar covariáveis medidas antes da exposição ou no contexto inicial.
COVARIAVEIS_PREVIAS = [
    "gravidade_clinica",
    "renda_pc_zirc",
    "idade",
    "regiao_zirconia",
    "unidade_zus",
    "capilaridade_defensoria",
    "completude_documental",
    "ano_pedido",
]

# Listar variáveis que devem existir na base analítica exportada.
VARIAVEIS_ANALITICAS = [
    "id_pedido",
    "assistencia_juridica",
    "indice_acesso_saude",
    "desfecho_pedido",
    "tempo_espera_dias",
    *COVARIAVEIS_PREVIAS,
]

# Registrar o dicionário também como DataFrame para auditoria computacional.
registros_dicionario = [
    {
        "nome": "id_pedido",
        "significado": "Identificador artificial único do pedido de acesso à saúde",
        "tipo": "texto",
        "escala": "identificador",
        "unidade": "não se aplica",
        "tipo_python": "string",
        "papel": "chave",
        "exemplos": "ZIR-000001; ZIR-000250",
    },
    {
        "nome": "assistencia_juridica",
        "significado": "Recebeu assistência jurídica institucional antes da decisão final",
        "tipo": "categórica binária",
        "escala": "nominal binária",
        "unidade": "Sim/Não",
        "tipo_python": "category",
        "papel": "independente principal",
        "exemplos": "Sim; Não",
    },
    {
        "nome": "indice_acesso_saude",
        "significado": "Escore composto de acesso efetivo ao cuidado em saúde",
        "tipo": "numérica contínua",
        "escala": "intervalar assumida",
        "unidade": "pontos de 0 a 100",
        "tipo_python": "float64",
        "papel": "dependente primária",
        "exemplos": "41.7; 68.2",
    },
    {
        "nome": "desfecho_pedido",
        "significado": "Resultado final observado na janela retrospectiva",
        "tipo": "categórica multinomial",
        "escala": "nominal com cinco níveis",
        "unidade": "categoria",
        "tipo_python": "category",
        "papel": "dependente secundária",
        "exemplos": "internacao_realizada; negado",
    },
    {
        "nome": "tempo_espera_dias",
        "significado": "Tempo entre protocolo e decisão ou atendimento",
        "tipo": "numérica contínua",
        "escala": "razão",
        "unidade": "dias",
        "tipo_python": "float64",
        "papel": "desfecho auxiliar/covariável descritiva",
        "exemplos": "12.5; 83.0",
    },
    {
        "nome": "gravidade_clinica",
        "significado": "Gravidade clínica registrada antes da assistência",
        "tipo": "categórica ordinal",
        "escala": "ordinal",
        "unidade": "nível 1 a 4",
        "tipo_python": "Int64 ou category ordenada",
        "papel": "confundidora observada",
        "exemplos": "1; 4",
    },
    {
        "nome": "renda_pc_zirc",
        "significado": "Renda familiar mensal per capita do requerente",
        "tipo": "numérica contínua",
        "escala": "razão",
        "unidade": "Z$ por pessoa/mês",
        "tipo_python": "float64",
        "papel": "confundidora observada",
        "exemplos": "180.50; 940.00",
    },
    {
        "nome": "idade",
        "significado": "Idade do requerente no protocolo",
        "tipo": "numérica discreta",
        "escala": "razão",
        "unidade": "anos completos",
        "tipo_python": "Int64",
        "papel": "confundidora observada",
        "exemplos": "34; 72",
    },
    {
        "nome": "regiao_zirconia",
        "significado": "Região administrativa da Zircônia",
        "tipo": "categórica multinomial",
        "escala": "nominal",
        "unidade": "região",
        "tipo_python": "category",
        "papel": "estrato/contexto",
        "exemplos": "R03; R17",
    },
    {
        "nome": "unidade_zus",
        "significado": "Unidade territorial do ZUS vinculada ao pedido",
        "tipo": "categórica multinomial",
        "escala": "nominal",
        "unidade": "unidade administrativa",
        "tipo_python": "category",
        "papel": "cluster inferencial",
        "exemplos": "UZ-014; UZ-087",
    },
    {
        "nome": "capilaridade_defensoria",
        "significado": "Presença local da Defensoria Pública antes do pedido",
        "tipo": "categórica ordinal",
        "escala": "ordinal",
        "unidade": "nível 0 a 3",
        "tipo_python": "Int64 ou category ordenada",
        "papel": "confundidora/contexto",
        "exemplos": "0; 3",
    },
    {
        "nome": "completude_documental",
        "significado": "Documentação inicial suficiente para análise do pedido",
        "tipo": "categórica binária",
        "escala": "nominal binária",
        "unidade": "Sim/Não",
        "tipo_python": "category",
        "papel": "covariável prévia",
        "exemplos": "Sim; Não",
    },
    {
        "nome": "ano_pedido",
        "significado": "Ano de protocolo do pedido",
        "tipo": "temporal discreta",
        "escala": "calendário",
        "unidade": "ano",
        "tipo_python": "Int64",
        "papel": "contexto temporal",
        "exemplos": "2023; 2025",
    },
    {
        "nome": "letramento_digital_latente",
        "significado": "Facilidade não observada de usar sistemas do ZUS e buscar canais institucionais",
        "tipo": "latente binária",
        "escala": "nominal binária",
        "unidade": "não observada",
        "tipo_python": "não exportar",
        "papel": "confundidora oculta",
        "exemplos": "0; 1 apenas no gerador",
    },
]
dicionario = pd.DataFrame(registros_dicionario)

# Validar unicidade e completude do dicionário.
assert dicionario["nome"].is_unique, "O dicionário contém nomes duplicados."
assert len(CATEGORIAS_DESFECHO) == 5, "O desfecho multinomial deve ter cinco níveis."
assert len(set(CATEGORIAS_DESFECHO)) == 5, "As categorias do desfecho devem ser distintas."
assert set(VARIAVEIS_ANALITICAS).issubset(set(dicionario["nome"])), (
    "Há variável analítica sem descrição no dicionário."
)
assert "assistencia_juridica" in VARIAVEIS_ANALITICAS, "Exposição ausente."
assert "indice_acesso_saude" in VARIAVEIS_ANALITICAS, "Dependente primária ausente."
assert "desfecho_pedido" in VARIAVEIS_ANALITICAS, "Dependente secundária ausente."
assert "letramento_digital_latente" not in VARIAVEIS_ANALITICAS, (
    "A latente não deve constar da base analítica."
)
assert set(COVARIAVEIS_PREVIAS).issubset(set(VARIAVEIS_ANALITICAS)), (
    "Covariáveis prévias devem estar no esquema analítico."
)

# Exigir preenchimento em todos os campos do quadro.
for coluna in dicionario.columns:
    valores = dicionario[coluna].astype(str).str.strip()
    assert valores.ne("").all(), f"Campo vazio encontrado em {coluna}."

# Exibir a versão computacional do dicionário.
display(dicionario)
print("Total de variáveis analíticas:", len(VARIAVEIS_ANALITICAS))
print("Categorias do desfecho:", ", ".join(CATEGORIAS_DESFECHO))

,nome,significado,tipo,escala,unidade,tipo_python,papel,exemplos
0,id_pedido,Identificador artificial único do pedido de ac...,texto,identificador,não se aplica,string,chave,ZIR-000001; ZIR-000250
1,assistencia_juridica,Recebeu assistência jurídica institucional ant...,categórica binária,nominal binária,Sim/Não,category,independente principal,Sim; Não
2,indice_acesso_saude,Escore composto de acesso efetivo ao cuidado e...,numérica contínua,intervalar assumida,pontos de 0 a 100,float64,dependente primária,41.7; 68.2
3,desfecho_pedido,Resultado final observado na janela retrospectiva,categórica multinomial,nominal com cinco níveis,categoria,category,dependente secundária,internacao_realizada; negado
4,tempo_espera_dias,Tempo entre protocolo e decisão ou atendimento,numérica contínua,razão,dias,float64,desfecho auxiliar/covariável descritiva,12.5; 83.0
5,gravidade_clinica,Gravidade clínica registrada antes da assistência,categórica ordinal,ordinal,nível 1 a 4,Int64 ou category ordenada,confundidora observada,1; 4
6,renda_pc_zirc,Renda familiar mensal per capita do requerente,numérica contínua,razão,Z$ por pessoa/mês,float64,confundidora observada,180.50; 940.00
7,idade,Idade do requerente no protocolo,numérica discreta,razão,anos completos,Int64,confundidora observada,34; 72
8,regiao_zirconia,Região administrativa da Zircônia,categórica multinomial,nominal,região,category,estrato/contexto,R03; R17
9,unidade_zus,Unidade territorial do ZUS vinculada ao pedido,categórica multinomial,nominal,unidade administrativa,category,cluster inferencial,UZ-014; UZ-087


Total de variáveis analíticas: 13
Categorias do desfecho: internacao_realizada, tratamento_ambulatorial, fornecimento_medicamento, negado, pendente_ou_sem_execucao


### Interpretação metodológica da seção

O dicionário explicita quais variáveis são observadas e quais pertencem apenas
ao mecanismo didático de simulação. Essa separação evita uma falha comum:
ajustar o modelo por uma variável que, na pesquisa real, não estaria disponível
ou seria medida depois da exposição.

Em uma adaptação com dados reais, o estudante deve acrescentar: fonte de cada
variável, responsável pela codificação, regras para valores ausentes, tratamento
de duplicidades, janela de aferição e justificativa para o papel analítico
assumido. O quadro também deve ser depositado no repositório antes da coleta,
para reduzir graus de liberdade analíticos e aumentar replicabilidade.